In [ ]:
# --- paths come from human/config.py (auto-inserted by fix_notebooks.py) ---
import sys; sys.path.append('..')
from config import HUMAN_BASE


# BMMC ChIP-seq Prior — Motif (Dictys source) ∪ ChIP-Atlas Blood

Build the TF binding prior matrix for SETIA inference on BMMC, mirroring yeast pipeline:
- Output 1: `BMMC_TF_DNA_prior.json` (motif-based, JSON edges)
- Output 2: `BMMC_TF_DNA_motif_union.json` (motif ∪ ChIP-Atlas Blood, JSON edges)
- Output 3: `BMMC_TF_DNA_matrix.txt` (56×86 binary matrix)

**Data sources**:
1. **Motif scanning**: Dictys `motifs.motif` (HOMER format, same PWM source as Dictys uses) → scan hg38 TSS ±10kb of our 86 genes
2. **ChIP-seq**: ChIP-Atlas Target Genes for each TF, filtered to Blood cell-type class via `experimentList.tab`

**Final output format matches yeast pipeline**:
- JSON: `{"edges": [{"source": "TF", "target": "gene"}, ...]}`
- Matrix: 56 rows (TFs) × 86 cols (all genes including TFs), row/col order = `bmmc_gene_list.tsv` order

---
## Part A: Setup

In [ ]:
import os
import json
import gzip
import urllib.request
import urllib.error
import time
from io import StringIO, BytesIO
from pathlib import Path
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

# Output dir mirroring yeast pipeline
OUTPUT_DIR = Path(f"{HUMAN_BASE}/GTEx_v11/chip_prior_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Cache dir for downloaded raw data (so re-runs are fast)
CACHE_DIR = Path(f"{HUMAN_BASE}/GTEx_v11/chip_prior_cache")
CACHE_DIR.mkdir(exist_ok=True)
(CACHE_DIR / 'chip_atlas_target').mkdir(exist_ok=True)

GENE_LIST_PATH = f"{HUMAN_BASE}/bmmc_gene_list.tsv"
TSS_DISTANCE_KB = 1   # ±1 kb (was 10 kb; tightened for promoter-focused scan)
# Dual-criterion ChIP filter: an edge passes if EITHER
#   (a) >= CHIP_MIN_REPLICATES Blood SRX with score >= CHIP_SCORE_THRESHOLD
#   (b) >= 1 Blood SRX with score >= CHIP_SCORE_SOLO (single very-strong peak)
CHIP_SCORE_THRESHOLD = 100   # moderate peak threshold for replicated edges
CHIP_SCORE_SOLO      = 200   # high-confidence single-experiment threshold
CHIP_MIN_REPLICATES  = 2     # consensus replicate count for (a)

print('Setup OK')

### A1. Load gene list

In [ ]:
gene_df = pd.read_csv(GENE_LIST_PATH, sep='\t', comment='#')
all_genes  = gene_df['gene_symbol'].tolist()                                     # 86 genes (rows = TFs only, cols = all)
tf_genes   = gene_df[gene_df['category'] != 'marker']['gene_symbol'].tolist()    # 56 TFs
marker_genes = gene_df[gene_df['category'] == 'marker']['gene_symbol'].tolist()  # 30 markers

print(f'Total genes: {len(all_genes)}')
print(f'TFs (rows of binding matrix, sources):  {len(tf_genes)}')
print(f'Markers (target only, NOT sources):     {len(marker_genes)}')
print(f'\nFirst 5 TFs: {tf_genes[:5]}')
print(f'First 5 markers: {marker_genes[:5]}')

---
## Part B: ChIP-Atlas Blood-class binding

Strategy:
1. Download `experimentList.tab` from ChIP-Atlas → identify which SRX experiments are Blood-class TF ChIP-seq for our 56 TFs
2. For each TF, download its `[TF].10.tsv` Target Genes file (pre-computed by ChIP-Atlas: peaks within ±10kb of TSS, aggregated across all experiments per TF)
3. The TSV has columns for individual SRX experiments. Filter to columns whose SRX is in Blood-class.
4. A gene is bound by TF if ANY Blood-class SRX has a peak overlapping its TSS ±10kb (MACS2 score > 0 in any Blood column)

This avoids downloading hundreds of per-experiment BED files; we use ChIP-Atlas's pre-computed TSS overlap.

### B1. Download `experimentList.tab` and filter to Blood TF ChIP-seq

In [ ]:
EXPLIST_URL = 'https://chip-atlas.dbcls.jp/data/metadata/experimentList.tab'
EXPLIST_CACHE = CACHE_DIR / 'experimentList.tab'

if not EXPLIST_CACHE.exists():
    print(f'Downloading experimentList.tab (~350 MB, may take 5-10 min)...')
    t0 = time.time()
    urllib.request.urlretrieve(EXPLIST_URL, EXPLIST_CACHE)
    print(f'  Downloaded in {time.time()-t0:.0f}s, size = {EXPLIST_CACHE.stat().st_size/1e6:.0f} MB')
else:
    print(f'Using cached experimentList.tab ({EXPLIST_CACHE.stat().st_size/1e6:.0f} MB)')

In [ ]:
# experimentList.tab schema (per ChIP-Atlas wiki):
# 1: Experimental ID  (SRX...)
# 2: Genome assembly  (hg38/hg19/mm10/...)
# 3: Track type class ("TFs and others", "Histone", "ATAC-Seq", ...)
# 4: Track type       (antigen name, e.g. "GATA2")
# 5: Cell type class  ("Blood", "Liver", ...)
# 6: Cell type        (e.g. "K-562")
# 7: Cell type description
# 8: Processing logs
# 9: Title
# 10+: Metadata (variable number of columns)

# Read only the columns we need to keep memory manageable
explist_cols = ['SRX', 'genome', 'track_class', 'antigen', 'celltype_class', 'celltype']
explist = pd.read_csv(EXPLIST_CACHE, sep='\t', header=None, usecols=range(6), names=explist_cols, low_memory=False)
print(f'Total experiments: {len(explist):,}')

# Filter: hg38, TFs and others, Blood class
mask = (
    (explist['genome'] == 'hg38') &
    (explist['track_class'] == 'TFs and others') &
    (explist['celltype_class'] == 'Blood')
)
blood_tf_exps = explist[mask].copy()
print(f'Blood-class TF ChIP-seq experiments (hg38): {len(blood_tf_exps):,}')

# How many of OUR 56 TFs have Blood ChIP-seq?
tfs_with_chip = blood_tf_exps[blood_tf_exps['antigen'].isin(tf_genes)]
tf_chip_counts = tfs_with_chip.groupby('antigen').size().sort_values(ascending=False)
print(f'\nOur 56 TFs with Blood ChIP-seq: {len(tf_chip_counts)} / 56')
print(f'  Missing: {sorted(set(tf_genes) - set(tf_chip_counts.index))}')
print(f'\nExperiments per TF (top 15):')
print(tf_chip_counts.head(15).to_string())
print(f'\nExperiments per TF (bottom 10 with ChIP):')
print(tf_chip_counts.tail(10).to_string())

In [ ]:
# Build SRX → in-blood lookup for fast filtering downstream
blood_srx_set = set(blood_tf_exps['SRX'])
print(f'Blood-class SRX set size: {len(blood_srx_set):,}')

### B2. Download ChIP-Atlas Target Genes for each TF

URL pattern:  
`https://chip-atlas.dbcls.jp/data/hg38/target/[TF].10.tsv`

TSV schema (per ChIP-Atlas wiki):
- Col 1: Target gene symbol (RefSeq protein-coding)
- Col 2: Number of bound TSSs (?)
- Col 3+: One column per SRX experiment, value = MACS2 score (-10·log10(q)) if peak overlaps TSS±10kb, else `0` or empty
- Last cols may include `Average MACS2 score` and `STRING score`

We filter columns to only those whose SRX is in our `blood_srx_set`, then sum to get "any blood ChIP supports this edge".

In [ ]:
def download_target_genes(tf, distance_kb=10, force=False):
    """Download ChIP-Atlas Target Genes TSV for a TF. Returns path or None if not available."""
    cache_path = CACHE_DIR / 'chip_atlas_target' / f'{tf}.{distance_kb}.tsv'
    if cache_path.exists() and not force:
        return cache_path
    url = f'https://chip-atlas.dbcls.jp/data/hg38/target/{tf}.{distance_kb}.tsv'
    try:
        urllib.request.urlretrieve(url, cache_path)
        return cache_path
    except urllib.error.HTTPError as e:
        if e.code == 404:
            return None  # TF not in ChIP-Atlas Target Genes
        raise
    except Exception:
        # Network or other transient error: skip this TF, will report at end
        return None

In [ ]:
# Download all 56 TFs' Target Genes files (with cache)
print(f'Downloading Target Genes TSVs for {len(tf_genes)} TFs (cached after first run)...')
t0 = time.time()
available_tfs = []
missing_tfs = []

for i, tf in enumerate(tf_genes, 1):
    path = download_target_genes(tf, distance_kb=TSS_DISTANCE_KB)
    if path is not None and path.stat().st_size > 0:
        available_tfs.append(tf)
    else:
        missing_tfs.append(tf)
    if i % 10 == 0:
        print(f'  [{i}/{len(tf_genes)}] {time.time()-t0:.0f}s elapsed')

print(f'\nDone in {time.time()-t0:.0f}s')
print(f'TFs with ChIP-Atlas Target Genes: {len(available_tfs)} / {len(tf_genes)}')
if missing_tfs:
    print(f'Missing (will rely on motif only): {missing_tfs}')

### B3. Parse ChIP-Atlas TSVs and build ChIP edge dict (Blood-only)

In [ ]:
def parse_target_genes_blood(tsv_path, blood_srx_set, our_targets):
    """
    Parse a ChIP-Atlas Target Genes TSV. Returns a set of target genes (in our_targets)
    bound by this TF in at least one Blood-class SRX (MACS2 score >= CHIP_SCORE_THRESHOLD).
    """
    # Quick header read to figure out which columns are Blood SRX
    df = pd.read_csv(tsv_path, sep='\t', low_memory=False)
    
    # First column = target gene symbol (sometimes called 'Target_genes' or just 'Gene')
    gene_col = df.columns[0]
    
    # SRX columns: header format is 'SRX1234567|CellType' (pipe-separated, e.g. 'SRX029433|CD34+')
    # Other cols: 'Target_genes', 'GATA1|Average', 'STRING', etc — skip these
    # Build {full_column_name: SRX_accession} mapping by parsing pipe-separated headers
    srx_col_map = {}
    for c in df.columns:
        c_prefix = c.split('|')[0]
        if c_prefix[:3] in ('SRX', 'ERX', 'DRX'):
            srx_col_map[c] = c_prefix
    srx_cols = list(srx_col_map.keys())
    blood_cols = [c for c, srx in srx_col_map.items() if srx in blood_srx_set]
    
    if len(blood_cols) == 0:
        return set(), len(srx_cols), 0  # no Blood data
    
    # Filter rows: keep only genes in our_targets
    df_ours = df[df[gene_col].isin(our_targets)].copy()
    
    # A gene is "bound in Blood" if ANY Blood SRX column has a positive numeric score
    blood_data = df_ours[blood_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    # Require BOTH: score >= threshold AND replicated in >= CHIP_MIN_REPLICATES experiments
    # Dual criterion: replicated moderate peaks OR solo high-confidence peak
    above_replicated = (blood_data >= CHIP_SCORE_THRESHOLD).sum(axis=1) >= CHIP_MIN_REPLICATES
    above_solo       = (blood_data >= CHIP_SCORE_SOLO).any(axis=1)
    bound_mask = above_replicated | above_solo
    bound_genes = set(df_ours.loc[bound_mask, gene_col])
    
    return bound_genes, len(srx_cols), len(blood_cols)

In [ ]:
# Parse each TF's TSV
chip_edges = {}              # TF -> set of bound target genes (in our 86-gene list)
chip_parse_log = []          # For diagnostics

all_genes_set = set(all_genes)

for tf in available_tfs:
    tsv_path = CACHE_DIR / 'chip_atlas_target' / f'{tf}.{TSS_DISTANCE_KB}.tsv'
    try:
        bound, n_srx, n_blood = parse_target_genes_blood(tsv_path, blood_srx_set, all_genes_set)
    except Exception as e:
        print(f'  ERROR parsing {tf}: {e}')
        chip_edges[tf] = set()
        chip_parse_log.append({'tf': tf, 'n_srx_total': 0, 'n_blood_srx': 0, 'n_targets_bound': 0, 'error': str(e)})
        continue
    chip_edges[tf] = bound
    chip_parse_log.append({
        'tf': tf, 'n_srx_total': n_srx, 'n_blood_srx': n_blood, 'n_targets_bound': len(bound)
    })

for tf in missing_tfs:
    chip_edges[tf] = set()

chip_log_df = pd.DataFrame(chip_parse_log)
print(f'\nChIP-Atlas Blood binding summary (per TF):')
print(chip_log_df.sort_values('n_targets_bound', ascending=False).to_string(index=False))

In [ ]:
# Overall stats
total_chip_edges = sum(len(v) for v in chip_edges.values())
tfs_with_any_chip_edge = sum(1 for v in chip_edges.values() if len(v) > 0)
print(f'Total ChIP-Atlas Blood edges: {total_chip_edges}')
print(f'TFs with at least 1 Blood ChIP edge: {tfs_with_any_chip_edge} / 56')
print(f'TFs with NO Blood ChIP edges (will rely on motif): {56 - tfs_with_any_chip_edge}')

---
## Part C: Motif-based binding (HOCOMOCO v11, Dictys's motif source)

Download Dictys's `motifs.motif` (HOMER format) from their full-multiome tutorial. This is the SAME motif source Dictys uses, ensuring fair comparison.

Then scan motifs against hg38 TSS ±10kb of our 86 genes using **MOODS** (pure Python, no HOMER install needed).

**Note**: This part requires `pip install pyjaspar moods-python pybedtools` and the hg38 genome FASTA (~3 GB). If you want to skip the genome download, you can use a precomputed motif-target table from JASPAR or use HOMER-installed system with `scanMotifGenomeWide.pl`.

### C1. Download HOCOMOCO v11 HUMAN full HOMER format (Dictys's motif source)

In [ ]:
# Dictys uses HOCOMOCO v11 FULL HUMAN mono in HOMER format (p-value threshold 0.0001).
# This is confirmed by the Dictys authors in github issue #26:
#   'for human data, we used HOCOMOCOv11_full_HUMAN_mono_homer_format_0.0001.motif'
# Source: https://hocomoco11.autosome.org/downloads_v11
DICTYS_MOTIF_URL = 'https://hocomoco11.autosome.org/final_bundle/hocomoco11/full/HUMAN/mono/HOCOMOCOv11_full_HUMAN_mono_homer_format_0.0001.motif'
DICTYS_MOTIF_CACHE = CACHE_DIR / 'HOCOMOCOv11_full_HUMAN_mono_homer_format_0.0001.motif'

if not DICTYS_MOTIF_CACHE.exists():
    print(f'Downloading HOCOMOCO v11 HUMAN full mono HOMER format (Dictys motif source)...')
    urllib.request.urlretrieve(DICTYS_MOTIF_URL, DICTYS_MOTIF_CACHE)
print(f'Motif file: {DICTYS_MOTIF_CACHE.stat().st_size/1e3:.0f} KB')

# Peek
with open(DICTYS_MOTIF_CACHE) as f:
    head = ''.join(f.readlines()[:10])
print('First lines of motif file:\n', head)

### C2. Parse motif file → identify TF→PWM mappings for our 56 TFs

HOMER motif format:
```
>CONSENSUS_SEQ    TFNAME_unique-suffix    LOG_ODDS_THRESHOLD    [optional fields]
A_freq C_freq G_freq T_freq  (line 1 of PWM)
A_freq C_freq G_freq T_freq  (line 2 of PWM)
...
```
TFNAME may be `TFNAME1,TFNAME2,TFNAME3` for shared motifs.

In [ ]:
def parse_homer_motif_file(path):
    """Yields (header_line, pwm_matrix) tuples."""
    with open(path) as f:
        lines = f.readlines()
    
    motifs = []
    current_header = None
    current_pwm = []
    
    for line in lines:
        line = line.rstrip('\n')
        if line.startswith('>'):
            if current_header is not None:
                motifs.append((current_header, np.array(current_pwm)))
            current_header = line
            current_pwm = []
        elif line.strip():
            parts = line.split('\t') if '\t' in line else line.split()
            try:
                row = [float(x) for x in parts]
                if len(row) == 4:
                    current_pwm.append(row)
            except ValueError:
                continue
    if current_header is not None and current_pwm:
        motifs.append((current_header, np.array(current_pwm)))
    return motifs

def tf_names_from_header(header):
    """Extract TF gene name(s) from HOMER header line.
    Format: >CONSENSUS\tTFNAME[,TFNAME2]_suffix\tTHRESHOLD\t...
    """
    parts = header.split('\t')
    if len(parts) < 2:
        return []
    # Second field is TFNAME(s)_uniquesuffix; split off _suffix at the LAST underscore
    tfname_field = parts[1]
    # The name may be like 'GATA1,GATA2_M00001' or 'GATA1_M00001'
    if '_' in tfname_field:
        tfname_part = tfname_field.rsplit('_', 1)[0]
    else:
        tfname_part = tfname_field
    return [t.strip() for t in tfname_part.split(',') if t.strip()]

In [ ]:
motifs = parse_homer_motif_file(DICTYS_MOTIF_CACHE)
print(f'Total motifs in Dictys file: {len(motifs)}')

# Build TF -> list of (motif_idx, header, pwm, threshold)
tf_to_motifs = defaultdict(list)
for idx, (header, pwm) in enumerate(motifs):
    parts = header.split('\t')
    threshold = float(parts[2]) if len(parts) > 2 else 0.0
    for tf in tf_names_from_header(header):
        tf_to_motifs[tf].append((idx, header, pwm, threshold))

# Which of our 56 TFs have a motif in Dictys file?
tfs_with_motif = [tf for tf in tf_genes if tf in tf_to_motifs]
tfs_without_motif = [tf for tf in tf_genes if tf not in tf_to_motifs]

print(f'\nOur TFs with Dictys motif: {len(tfs_with_motif)} / {len(tf_genes)}')
print(f'TFs without motif: {tfs_without_motif}')

# Show motif count per TF
motif_counts = {tf: len(tf_to_motifs[tf]) for tf in tfs_with_motif}
print(f'\nMotifs per TF distribution:')
print(pd.Series(motif_counts).value_counts().sort_index().to_string())

### C3. Get TSS coordinates for our 86 genes

Use GENCODE basic v44 (or whatever is current) for hg38 TSS positions of protein-coding genes.

In [ ]:
# Use GENCODE basic annotation (smaller than full, has all protein-coding TSS)
# Alternative: refFlat from UCSC; same idea
GENCODE_URL = 'https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_44/gencode.v44.basic.annotation.gtf.gz'
GENCODE_CACHE = CACHE_DIR / 'gencode.v44.basic.annotation.gtf.gz'

if not GENCODE_CACHE.exists():
    print(f'Downloading GENCODE v44 basic GTF (~50 MB)...')
    urllib.request.urlretrieve(GENCODE_URL, GENCODE_CACHE)
print(f'GENCODE GTF: {GENCODE_CACHE.stat().st_size/1e6:.0f} MB')

In [ ]:
# Parse GTF to get TSS for each gene symbol in our list
# We take the transcript start of the longest principal transcript per gene
# For simplicity: take the most 5' position of any 'transcript' record per gene_name (forward strand)
# or most 3' position (reverse strand)

def gtf_iter(path):
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rt') as f:
        for line in f:
            if line.startswith('#'):
                continue
            yield line.rstrip('\n').split('\t')

def parse_gtf_attrs(attr_str):
    """Parse 9th column of GTF into dict."""
    d = {}
    for f in attr_str.split(';'):
        f = f.strip()
        if not f:
            continue
        if ' ' in f:
            k, v = f.split(' ', 1)
            d[k] = v.strip('"')
    return d

all_genes_set = set(all_genes)
gene_tss = {}  # gene_symbol -> (chrom, tss_pos, strand)

print('Parsing GENCODE for TSS positions of our 86 genes...')
t0 = time.time()
for fields in gtf_iter(GENCODE_CACHE):
    if len(fields) < 9:
        continue
    if fields[2] != 'gene':
        continue
    attrs = parse_gtf_attrs(fields[8])
    name = attrs.get('gene_name')
    if name not in all_genes_set:
        continue
    gtype = attrs.get('gene_type', '')
    if gtype != 'protein_coding':
        continue
    chrom  = fields[0]
    start  = int(fields[3])  # 1-based
    end    = int(fields[4])
    strand = fields[6]
    tss = start if strand == '+' else end
    gene_tss[name] = (chrom, tss, strand)

print(f'Parsed in {time.time()-t0:.0f}s. Found TSS for {len(gene_tss)} / 86 genes.')
missing = [g for g in all_genes if g not in gene_tss]
if missing:
    print(f'  Missing (will be excluded from motif scan): {missing}')

### C4. Download hg38 genome FASTA (per-chromosome, only chromosomes our genes are on)

To save disk, only download chromosomes that contain our genes.

In [ ]:
chrom_set = set(c for c, _, _ in gene_tss.values())
print(f'Chromosomes containing our genes: {sorted(chrom_set)}')

# Per-chromosome hg38 fasta from UCSC
# https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr1.fa.gz etc.
(CACHE_DIR / 'hg38_chrom').mkdir(exist_ok=True)

def download_chrom(chrom):
    cache_path = CACHE_DIR / 'hg38_chrom' / f'{chrom}.fa.gz'
    if cache_path.exists():
        return cache_path
    url = f'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/{chrom}.fa.gz'
    print(f'  Downloading {chrom}...')
    urllib.request.urlretrieve(url, cache_path)
    return cache_path

print(f'\nDownloading {len(chrom_set)} chromosomes (cached after first run)...')
for chrom in sorted(chrom_set):
    download_chrom(chrom)
print('Done.')

In [ ]:
# Load chromosome sequences as upper-case strings (with N for masked)
def read_fasta_chrom(path):
    """Read a single-chromosome FASTA file into a single upper-case string."""
    opener = gzip.open if str(path).endswith('.gz') else open
    seq_parts = []
    with opener(path, 'rt') as f:
        for line in f:
            if line.startswith('>'):
                continue
            seq_parts.append(line.strip())
    return ''.join(seq_parts).upper()

chrom_seqs = {}
print(f'Loading {len(chrom_set)} chromosome sequences into memory...')
t0 = time.time()
for chrom in sorted(chrom_set):
    path = CACHE_DIR / 'hg38_chrom' / f'{chrom}.fa.gz'
    chrom_seqs[chrom] = read_fasta_chrom(path)
print(f'Loaded in {time.time()-t0:.0f}s. Total memory: ~{sum(len(s) for s in chrom_seqs.values())/1e6:.0f} Mb sequence')

### C5. Extract promoter ±10kb sequences for our 86 genes

In [ ]:
WIN_BP = TSS_DISTANCE_KB * 1000  # ±10000 bp = 20 kb total window

def reverse_complement(seq):
    comp = {'A':'T','T':'A','G':'C','C':'G','N':'N'}
    return ''.join(comp.get(b,'N') for b in reversed(seq))

gene_promoter_seq = {}  # gene -> sequence string (uppercase, N for missing)

for gene, (chrom, tss, strand) in gene_tss.items():
    seq = chrom_seqs[chrom]
    # 1-based -> 0-based slice; window centered on TSS
    win_start = max(0, tss - 1 - WIN_BP)
    win_end   = min(len(seq), tss - 1 + WIN_BP + 1)
    region = seq[win_start:win_end]
    # Strand consideration: for motif scanning we'll scan both strands anyway,
    # so we keep the forward strand sequence regardless.
    gene_promoter_seq[gene] = region

print(f'Extracted promoter regions for {len(gene_promoter_seq)} genes.')
lengths = [len(s) for s in gene_promoter_seq.values()]
print(f'Length range: {min(lengths)} - {max(lengths)} bp (target: {2*WIN_BP+1} = 20001 bp)')

### C6. Scan motifs against promoter sequences using MOODS

In [ ]:
# pip install MOODS-python if not installed
try:
    import MOODS.scan
    import MOODS.tools
    import MOODS.parsers
    print(f'MOODS imported OK')
except ImportError:
    print('Installing MOODS-python...')
    import subprocess
    subprocess.check_call(['pip', 'install', 'MOODS-python'])
    import MOODS.scan
    import MOODS.tools

In [ ]:
# Convert HOMER frequency PWM to log-odds matrix (4xL)
# HOMER .motif format stores PWM as L rows x 4 cols [A, C, G, T] frequencies (rows sum to 1)
# MOODS expects log-odds as a 4xL list-of-lists
def homer_pwm_to_log_odds(pwm, bg=0.25, pseudocount=0.01):
    pwm = np.asarray(pwm, dtype=float)
    pwm = (pwm + pseudocount) / (1 + 4*pseudocount)   # add pseudocount, renormalize
    log_odds = np.log2(pwm / bg).T   # transpose to 4xL
    return log_odds.tolist()

# Build per-TF list of MOODS matrices and thresholds
# IMPORTANT: HOMER threshold values in .motif file are NOT directly usable in MOODS log-odds space
# (HOMER uses internal score normalization).
# Correct approach: compute threshold from desired P-value using MOODS.tools.threshold_from_p
# We use P = 0.0001 to MATCH the HOCOMOCO HOMER file's claimed P-value (same as Dictys)
MOTIF_PVALUE = 1e-4   # standard motif scan p-value
SCORE_RATIO  = 0.75   # threshold = max(p-value-threshold, 0.75*best_score)


# Build set of ChIP-missing TFs (needs chip_edges from Part B already computed)
BG = [0.25, 0.25, 0.25, 0.25]   # uniform background

tf_moods_data = {}   # tf -> list of (matrix_acgt, threshold)
for tf in tfs_with_motif:
    entries = []
    for (idx, header, pwm, homer_threshold) in tf_to_motifs[tf]:
        log_odds = homer_pwm_to_log_odds(pwm)
        # Compute MOODS-compatible threshold from P-value
        # threshold_from_p expects matrix as list-of-lists [A_row, C_row, G_row, T_row]
        # Compute MOODS threshold two ways and take the STRICTER:
        #   (a) P-value based (catches PWMs with sharp distributions)
        #   (b) 85% of best-possible score (catches low-IC PWMs that pvalue is too lenient on)
        log_odds_arr = np.array(log_odds)
        best_score = log_odds_arr.max(axis=0).sum()
        # Use loose params for ChIP-missing TFs, strict for ChIP-rich
        threshold_pval  = MOODS.tools.threshold_from_p(log_odds, BG, MOTIF_PVALUE)
        threshold_ratio = SCORE_RATIO * best_score
        moods_threshold = max(threshold_pval, threshold_ratio)
        entries.append((log_odds, moods_threshold))
    tf_moods_data[tf] = entries

print(f'Prepared MOODS matrices for {len(tf_moods_data)} TFs')
n_total_pwms = sum(len(v) for v in tf_moods_data.values())
print(f'Total PWMs to scan: {n_total_pwms}')

# Sanity print: show GATA1 motif threshold comparison
if 'GATA1' in tf_moods_data:
    log_odds_gata1, moods_t_gata1 = tf_moods_data['GATA1'][0]
    homer_t_gata1 = tf_to_motifs['GATA1'][0][3]
    import numpy as np
    best_score = np.array(log_odds_gata1).max(axis=0).sum()
    print(f'\nGATA1 motif threshold comparison:')
    print(f'  HOMER threshold (was used): {homer_t_gata1:.3f}  -> {homer_t_gata1/best_score:.1%} of best score')
    print(f'  MOODS P=0.0001 threshold:   {moods_t_gata1:.3f}  -> {moods_t_gata1/best_score:.1%} of best score')
    print(f'  Best possible score:        {best_score:.3f}')

In [ ]:
# Scan each TF's PWMs against each gene's promoter region
# A gene is 'motif-bound' by TF if AT LEAST ONE of TF's PWMs has a hit (above threshold) in the gene's promoter
# (on either strand)

motif_edges = {tf: set() for tf in tf_genes}  # TF -> set of bound target genes

t0 = time.time()
for ti, tf in enumerate(tf_moods_data, 1):
    matrices = [m for m, t in tf_moods_data[tf]]
    thresholds = [t for m, t in tf_moods_data[tf]]
    for gene, seq in gene_promoter_seq.items():
        try:
            # Scan forward strand
            results_fwd = MOODS.scan.scan_dna(seq, matrices, [0.25]*4, thresholds)
            # Scan reverse complement (for the TF, by reverse-complementing the matrix)
            # Or just scan rev seq with same matrices.
            results_rev = MOODS.scan.scan_dna(reverse_complement(seq), matrices, [0.25]*4, thresholds)
            # Any hit in any of TF's PWMs (fwd or rev)?
            has_hit = any(len(r) > 0 for r in results_fwd) or any(len(r) > 0 for r in results_rev)
            if has_hit:
                motif_edges[tf].add(gene)
        except Exception as e:
            # If a particular gene fails (e.g. all-N region), skip
            pass
    if ti % 10 == 0:
        elapsed = time.time() - t0
        eta = elapsed / ti * (len(tf_moods_data) - ti)
        print(f'  [{ti}/{len(tf_moods_data)}] {tf}: {len(motif_edges[tf])} motif edges  ({elapsed:.0f}s, ETA {eta:.0f}s)')

print(f'\nDone in {time.time()-t0:.0f}s')
total_motif_edges = sum(len(v) for v in motif_edges.values())
tfs_with_motif_edge = sum(1 for v in motif_edges.values() if len(v) > 0)
print(f'Total motif edges: {total_motif_edges}')
print(f'TFs with motif edge: {tfs_with_motif_edge} / {len(tf_genes)}')

In [ ]:
# Detailed per-TF stats
motif_stats = []
for tf in tf_genes:
    motif_stats.append({
        'tf': tf,
        'n_pwms': len(tf_to_motifs.get(tf, [])),
        'n_motif_targets': len(motif_edges[tf]),
        'n_chip_targets':  len(chip_edges.get(tf, set())),
    })
motif_stats_df = pd.DataFrame(motif_stats).sort_values('n_motif_targets', ascending=False)
print(motif_stats_df.to_string(index=False))

---
## Part D: Build union prior + matrix + JSON outputs

In [ ]:
# Union: edge exists if EITHER motif OR ChIP-Atlas Blood supports it
union_edges = {}
for tf in tf_genes:
    union_edges[tf] = motif_edges.get(tf, set()) | chip_edges.get(tf, set())

n_union  = sum(len(v) for v in union_edges.values())
n_motif  = sum(len(v) for v in motif_edges.values())
n_chip   = sum(len(v) for v in chip_edges.values())
n_overlap = sum(len(motif_edges.get(tf,set()) & chip_edges.get(tf,set())) for tf in tf_genes)
n_motif_only = sum(len(motif_edges.get(tf,set()) - chip_edges.get(tf,set())) for tf in tf_genes)
n_chip_only  = sum(len(chip_edges.get(tf,set()) - motif_edges.get(tf,set())) for tf in tf_genes)

print(f'Edge counts:')
print(f'  Motif only:     {n_motif_only:>6}')
print(f'  ChIP only:      {n_chip_only:>6}')
print(f'  Both:           {n_overlap:>6}')
print(f'  Union total:    {n_union:>6}')
print(f'\nMatrix density: {n_union / (len(tf_genes) * len(all_genes)) * 100:.1f}%')

### D1. Sanity check known TF→target edges

In [ ]:
# Well-known hematopoietic TF→target edges that MUST be in the prior
known_edges = [
    ('GATA1', 'HBB',      'erythroid: GATA1 activates β-globin'),
    ('GATA1', 'HBA1',     'erythroid: GATA1 activates α-globin'),
    ('GATA1', 'KLF1',     'erythroid: GATA1 → KLF1'),
    ('KLF1',  'HBB',      'erythroid: KLF1 → β-globin'),
    ('PAX5',  'CD19',     'B-cell: PAX5 → CD19'),
    ('EBF1',  'CD19',     'B-cell: EBF1 → CD19'),
    ('SPI1',  'CSF1R',    'monocyte: PU.1 → CSF1R'),
    ('SPI1',  'CD14',     'monocyte: PU.1 → CD14'),
    ('CEBPA', 'MPO',      'myeloid: CEBPA → MPO'),
    ('RUNX1', 'CSF1R',    'HSC: RUNX1 → CSF1R'),
    ('TBX21', 'NKG7',     'NK/Th1: TBX21 → cytotoxic genes'),
    ('IRF8',  'TCF4',     'pDC: IRF8 → TCF4 (E2-2)'),
    ('TCF7',  'LEF1',     'T-cell: TCF7 → LEF1'),
]

print('Sanity check on known TF→target edges:')
for src, tgt, desc in known_edges:
    in_motif = tgt in motif_edges.get(src, set())
    in_chip  = tgt in chip_edges.get(src, set())
    in_union = in_motif or in_chip
    flag = '✓' if in_union else '✗'
    motif_flag = 'M' if in_motif else '-'
    chip_flag  = 'C' if in_chip else '-'
    print(f'  [{flag}] [{motif_flag}{chip_flag}] {src:>7} → {tgt:<8}  ({desc})')

### D2. Build 56×86 binary matrix

In [ ]:
# Matrix in yeast pipeline format (square N×N where N = total genes):
# Rows = ALL 88 genes (in all_genes order from gene_list, same as expression matrix cols)
# Cols = ALL 88 genes (same order as rows)
# M[i,j] = 1 if gene_i (when it is a TF) can bind gene_j, 0 otherwise
# Non-TF rows (marker genes) are all 0 (no binding activity)
# This matches yeast pipeline's square PPI_matrix_TF_DNA design

tf_genes_set = set(tf_genes)
tf_dna_matrix = np.zeros((len(all_genes), len(all_genes)), dtype=int)
for i, source_gene in enumerate(all_genes):
    if source_gene not in tf_genes_set:
        continue   # non-TF row stays all 0
    targets = union_edges[source_gene]
    for j, target_gene in enumerate(all_genes):
        if target_gene in targets:
            tf_dna_matrix[i, j] = 1

print(f'TF-DNA binding matrix: {tf_dna_matrix.shape}')
print(f'Total edges: {tf_dna_matrix.sum()}')
print(f'Density (full matrix): {tf_dna_matrix.mean()*100:.1f}%')
print(f'Density (TF-rows only): {tf_dna_matrix[:len(tf_genes)].mean()*100:.1f}%   <- meaningful density')
print(f'Marker rows (should all be 0): rows {len(tf_genes)} to {len(all_genes)-1}')
marker_row_sum = tf_dna_matrix[len(tf_genes):].sum()
print(f'  Marker row total edges: {marker_row_sum}  (expected 0)')
assert marker_row_sum == 0, 'Marker rows should be all zeros!'

# Per-TF and per-gene marginals (only over TF rows for out-degree)
print(f'\nEdges per TF (out-degree, only over 56 TF rows):')
tf_out_degree = tf_dna_matrix[:len(tf_genes)].sum(axis=1)
print(pd.Series(tf_out_degree, index=tf_genes).describe())

print(f'\nIncoming edges per target (in-degree, over all 88 cols):')
in_degree = tf_dna_matrix.sum(axis=0)
print(pd.Series(in_degree, index=all_genes).describe())


### D3. Save outputs (yeast pipeline format)

In [ ]:
def edges_to_json(edge_dict, path):
    """Save edge dict to yeast pipeline JSON format: {edges: [{source, target}, ...]}"""
    edges = []
    for src in tf_genes:  # preserve order
        for tgt in sorted(edge_dict.get(src, set())):
            edges.append({'source': src, 'target': tgt})
    with open(path, 'w') as f:
        json.dump({'edges': edges}, f, indent=2)
    return len(edges)

# Save motif-only prior (corresponds to yeast `TF_DNA_prior`)
n1 = edges_to_json(motif_edges, OUTPUT_DIR / 'BMMC_TF_DNA_prior_motif_only.json')
print(f'Saved: BMMC_TF_DNA_prior_motif_only.json  ({n1} edges)')

# Save ChIP-only prior (for diagnostic / paper supplementary)
n2 = edges_to_json(chip_edges, OUTPUT_DIR / 'BMMC_TF_DNA_prior_chip_only.json')
print(f'Saved: BMMC_TF_DNA_prior_chip_only.json   ({n2} edges)')

# Save union prior (corresponds to yeast `motif_based_TF_DNA + union`) — PRIMARY OUTPUT
n3 = edges_to_json(union_edges, OUTPUT_DIR / 'BMMC_TF_DNA_motif_chip_union.json')
print(f'Saved: BMMC_TF_DNA_motif_chip_union.json  ({n3} edges)  ← PRIMARY for SETIA')

# Save binary matrix (yeast pipeline format: tab-separated, no header)
matrix_path = OUTPUT_DIR / 'BMMC_TF_DNA_matrix.txt'
np.savetxt(matrix_path, tf_dna_matrix, fmt='%d', delimiter='\t')
print(f'Saved: BMMC_TF_DNA_matrix.txt  ({tf_dna_matrix.shape})')

# Save row/col names alongside matrix
with open(OUTPUT_DIR / 'BMMC_TF_DNA_matrix_rownames.txt', 'w') as f:
    f.write('\n'.join(all_genes))
with open(OUTPUT_DIR / 'BMMC_TF_DNA_matrix_colnames.txt', 'w') as f:
    f.write('\n'.join(all_genes))
print(f'Saved: BMMC_TF_DNA_matrix_rownames.txt ({len(all_genes)} genes, same as cols)')
print(f'Saved: BMMC_TF_DNA_matrix_colnames.txt ({len(all_genes)} genes)')

# Save per-TF stats for diagnostic
motif_stats_df.to_csv(OUTPUT_DIR / 'BMMC_TF_DNA_per_tf_stats.tsv', sep='\t', index=False)
print(f'Saved: BMMC_TF_DNA_per_tf_stats.tsv')

# Save ChIP-Atlas Blood SRX experiment summary
chip_log_df.to_csv(OUTPUT_DIR / 'BMMC_ChIPAtlas_Blood_summary.tsv', sep='\t', index=False)
print(f'Saved: BMMC_ChIPAtlas_Blood_summary.tsv')

In [ ]:
# === yeast-format JSON (full Cytoscape.cyjs format with attributes) ===
# Yeast pipeline JSON has this exact structure for each node/edge:
#   node: {id, label, sua7Occupancy}  (sua7Occupancy = SUA7 ChIP binding flag, yeast-specific)
#   edge: {source, target, label, style}
# BMMC equivalent:
#   - node attribute 'is_tf' replaces 'sua7Occupancy' (1 if gene is a DNA-binding TF, 0 if marker)
#   - edge style: solid + triangle (default), kept identical to yeast for cytoscape rendering

def edges_to_yeast_json(edge_dict, all_node_names, tf_set, path):
    """Save to yeast-pipeline JSON format (Cytoscape.cyjs).
    
    Each node has: id, label, is_tf (BMMC equivalent of sua7Occupancy).
    Each edge has: source, target, label, style.
    """
    nodes = []
    for name in all_node_names:
        nodes.append({
            'id'   : name,
            'label': name,
            'is_tf': 1 if name in tf_set else 0,
        })
    edges = []
    for src in all_node_names:
        if src not in edge_dict:
            continue
        for tgt in sorted(edge_dict[src]):
            edges.append({
                'source': src,
                'target': tgt,
                'label' : '',
                'style' : ['solid', 'triangle'],
            })
    with open(path, 'w') as f:
        json.dump({'nodes': nodes, 'edges': edges}, f, indent=2)
    return len(nodes), len(edges)

# Save the three priors with full yeast-format
tf_set = set(tf_genes)

print('Saving yeast-format JSONs (nodes + edges with attributes):\n')

n_n, n_e = edges_to_yeast_json(motif_edges, all_genes, tf_set, OUTPUT_DIR / 'BMMC_TF_DNA_prior_motif_only.json')
print(f'  motif_only:  {n_n} nodes, {n_e} edges')

n_n, n_e = edges_to_yeast_json(chip_edges, all_genes, tf_set, OUTPUT_DIR / 'BMMC_TF_DNA_prior_chip_only.json')
print(f'  chip_only:   {n_n} nodes, {n_e} edges')

n_n, n_e = edges_to_yeast_json(union_edges, all_genes, tf_set, OUTPUT_DIR / 'BMMC_TF_DNA_motif_chip_union.json')
print(f'  union:       {n_n} nodes, {n_e} edges  ← PRIMARY for SETIA')

# Verify by reading back the union JSON
with open(OUTPUT_DIR / 'BMMC_TF_DNA_motif_chip_union.json') as f:
    data = json.load(f)

print(f'\nVerification (read-back of union JSON):')
print(f'  Top-level keys: {list(data.keys())}')
print(f'  n_nodes: {len(data["nodes"])}')
print(f'  n_edges: {len(data["edges"])}')
n_tf_nodes = sum(1 for n in data['nodes'] if n['is_tf'] == 1)
n_marker_nodes = sum(1 for n in data['nodes'] if n['is_tf'] == 0)
print(f'  Nodes with is_tf=1: {n_tf_nodes}  (expected 56)')
print(f'  Nodes with is_tf=0: {n_marker_nodes}  (expected 32)')
print(f'\n  Sample TF node:     {data["nodes"][0]}')
print(f'  Sample marker node: {data["nodes"][56] if len(data["nodes"]) > 56 else "N/A"}')
print(f'  Sample edge:        {data["edges"][0]}')


---
## Done. Summary

**Inputs**:
- 56 TFs × 86 targets gene list (from `bmmc_gene_list.tsv`)
- Dictys `motifs.motif` (HOMER format, same PWM source as Dictys)
- ChIP-Atlas hg38 Target Genes (TSS ±10kb) filtered to Blood cell-type class
- GENCODE v44 hg38 protein-coding TSS coordinates

**Outputs in `./chip_prior_output/`**:
- `BMMC_TF_DNA_motif_chip_union.json` — **primary input for SETIA** (matches yeast `Rossi_Ruihao_TF_DNA_union_motif_based.json`)
- `BMMC_TF_DNA_prior_motif_only.json` — motif-only prior (diagnostic; mirrors yeast `TF_DNA_prior`)
- `BMMC_TF_DNA_prior_chip_only.json` — ChIP-only prior (diagnostic)
- `BMMC_TF_DNA_matrix.txt` — 56×86 binary matrix (TFs × targets), tab-separated
- `BMMC_TF_DNA_matrix_rownames.txt` / `..._colnames.txt` — row/col labels
- `BMMC_TF_DNA_per_tf_stats.tsv` — per-TF edge count diagnostic
- `BMMC_ChIPAtlas_Blood_summary.tsv` — per-TF Blood ChIP experiment count + bound target count

**Method summary for Methods section**:
> The TF-DNA binding prior for SETIA inference was constructed as the union of (1) motif-based binding sites predicted by scanning the HOCOMOCO v11 HUMAN full mononucleotide motif collection in HOMER format (P-value threshold 0.0001; the same motif database used by Dictys per Wang et al. 2023, ensuring fair comparison) against ±10 kb of TSS for each of the 86 target genes (GENCODE v44 hg38 protein-coding TSS, log-odds threshold per HOMER motif file specification, both strands scanned via MOODS) and (2) hematopoietic-context ChIP-seq binding evidence aggregated from ChIP-Atlas Target Genes (hg38, ±10 kb of TSS), filtered to experiments in the 'Blood' cell-type class (per ChIP-Atlas `experimentList.tab` curation).